# Splitting the Dataset

For the beginning, I have decided to go for a 80/10/10 split for train/val/test

## 1.Imports and File Directories

In [23]:
import h5py
import numpy as np

DATA_DIR           = '../data/'
DATA_RAW_DIR        = DATA_DIR + 'raw/'
DATA_PROCESSED_DIR  = DATA_DIR + 'processed/'
DATA_SPLITS_DIR  = DATA_DIR + 'splits/'

RMA_CLEAN_DIR       = DATA_PROCESSED_DIR + 'Rma_clean.h5'
UMI_CLEAN_DIR       = DATA_PROCESSED_DIR + 'Umi_clean.h5'

## 2. Stratify Function

In [24]:
def stratified_split_indices(mods, snrs, domain, 
                             train_frac = 0.8, 
                             val_frac = 0.1, 
                             test_frac = 0.1, 
                             seed = 42):
    """
    Returns (train_idx, val_idx, test_idx) such that every (mod, SNR, domain)
    group is split in the same proportions -- preserving class + SNR + domain
    balance across all three splits.
    """

    # This line is to sanity check all thrree fractions actually do
    # Add up to %100.
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6

    rng = np.random.default_rng(seed)
    mod_idx = np.argmax(mods, axis = 1)

    train_idx, val_idx, test_idx = [], [], []

    keys = np.stack([mod_idx, snrs, domain], axis=1)
    unique_keys, inverse = np.unique(keys, axis=0, return_inverse = True)

    for g in range(len(unique_keys)):

        group_rows = np.where(inverse == g)[0]
        rng.shuffle(group_rows)

        n = len(group_rows)
        n_train = int(n * train_frac)
        n_val   = int(n * val_frac)

        train_idx.append(group_rows[:n_train])
        val_idx.append(group_rows[n_train:n_train + n_val])
        test_idx.append(group_rows[n_train + n_val:])

    return (
        np.concatenate(train_idx),
        np.concatenate(val_idx),
        np.concatenate(test_idx)
        )

## 3. Opening the h5 files 

In [25]:
with h5py.File(RMA_CLEAN_DIR, 'r') as f:
    rma_data, rma_mods, rma_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
rma_domain = np.zeros(len(rma_snrs), dtype = np.int8)

with h5py.File(UMI_CLEAN_DIR, 'r') as f:
    umi_data, umi_mods, umi_snrs = f['Data'][:], f['Mods'][:], f['SNRs'][:]
umi_domain = np.ones(len(umi_snrs), dtype = np.int8)

# Pooling all into one dataset
all_data   = np.concatenate([rma_data, umi_data])
all_mods   = np.concatenate([rma_mods, umi_mods])
all_snrs   = np.concatenate([rma_snrs, umi_snrs])
all_domain = np.concatenate([rma_domain, umi_domain])

print(f"Combined dataset: {all_data.shape[0]} rows")

# Free Memory
del rma_data, rma_mods, rma_snrs, umi_data, umi_mods, umi_snrs

Combined dataset: 163840 rows


## 4. Using the Stratify Function

In [26]:
train_idx, val_idx, test_idx = stratified_split_indices(
    all_mods, all_snrs, all_domain,
    train_frac = 0.8, val_frac = 0.1, test_frac = 0.1, seed = 42
)

print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

Train: 131040  Val: 16320  Test: 16480


## 5. Sanity Check

confirm domain ratio is preserved in each split

In [27]:
for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    ratio = all_domain[idx].mean()
    print(f"{name}: {ratio:.3f} fraction Umi (should be ~0.5)")

train: 0.500 fraction Umi (should be ~0.5)
val: 0.500 fraction Umi (should be ~0.5)
test: 0.500 fraction Umi (should be ~0.5)


## 6. Saving the Files

In [28]:
def save_split(path, idx):
    with h5py.File(path, 'w') as f:
        f.create_dataset('Data',   data=all_data[idx])
        f.create_dataset('Mods',   data=all_mods[idx])
        f.create_dataset('SNRs',   data=all_snrs[idx])
        f.create_dataset('Domain', data=all_domain[idx])  # 0 = Rma, 1 = Umi

In [29]:
save_split(DATA_SPLITS_DIR + 'train.h5', train_idx)
save_split(DATA_SPLITS_DIR + 'val.h5',   val_idx)
save_split(DATA_SPLITS_DIR + 'test.h5',  test_idx)

print("Saved train.h5, val.h5, test.h5 to", DATA_SPLITS_DIR)

Saved train.h5, val.h5, test.h5 to ../data/splits/
